# image_new: grade from masked-circle crops, balanced by augmentation, tuned transfer learning (Colab, GPU)

* **Crops:** the grey-masked circle (whole liquor, outside grey), `image/preprocess_images.py --crop masked`.
* **Augmentation** (`augmentation.py`): rotation and flip only. The grades with fewer training photos get augmented copies until every grade has as many training photos as the largest (BOPF). Copies are made only from each split's training photos; validation and test keep only real photos.
* **Models** (one file each in `models/`): EfficientNet-B0, ResNet18, MobileNetV2.
* **Before vs after tuning:** default hyperparameters (the official image recipe) vs the best of 12 random trials per model, chosen on validation data only (seeds 42-44), each retrained on all 10 seeds.
* **Learning curve:** training samples vs accuracy for the best tuned model.

Grade only. **Runtime: GPU.** **First run:** `FAST = True` (smoke test, about 20 min). **Real run:** `FAST = False`, about 6 GPU-hours in total; every step resumes after a disconnect (run the cell again).

## 1. Mount Google Drive

If `drive.mount` fails in VS Code, run *Colab: Mount Google Drive to Server* from the command palette instead.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

%cd /content/drive/MyDrive/TeaProject

## 2. Find the project and the dataset, check the GPU

`FAST = True` = smoke test (2 seeds, 1 epoch, 3 photos per sample, 2 tuning trials, 2 curve points); its results are meaningless.

In [ ]:
import os
from pathlib import Path

DRIVE_ROOT = Path('/content/drive/MyDrive/TeaProject')
DATA_ROOT = DRIVE_ROOT / 'dataset'
FAST = True
FRESH = False  # True = retrain every unit instead of reusing finished ones

candidates = [DRIVE_ROOT / 'Tea_Multimodal_Classifier_Claude', DRIVE_ROOT]
PROJECT_DIR = next((p for p in candidates if (p / 'unimodels' / 'image_new' / 'train_image_new.py').exists()), None)
if PROJECT_DIR is None:
    raise FileNotFoundError(f'unimodels/image_new/train_image_new.py not found in {DRIVE_ROOT}. '
                            'Upload the unimodels/image_new files first.')
if not (DATA_ROOT / 'Raw Dataset').exists():
    raise FileNotFoundError(f'Raw Dataset not found in {DATA_ROOT}')
os.chdir(PROJECT_DIR)

ROOT = Path('/content/image_new_smoke') if FAST else PROJECT_DIR / 'unimodels' / 'image_new'
OUT_DEFAULT, OUT_TUNED = ROOT / 'results_default', ROOT / 'results_tuned'
TUNING, LC, CMP, AUG = (ROOT / 'results' / x for x in ('tuning', 'learning_curve', 'comparison', 'augmentation'))
PREP = PROJECT_DIR / 'unimodels' / 'image' / 'results' / 'preprocess_masked'
FASTFLAG = '--fast' if FAST else ''
TRAINFLAGS = FASTFLAG + (' --fresh' if FRESH else '')
SEEDS = '42 43' if FAST else '42 43 44 45 46 47 48 49 50 51'

import torch
gpu = torch.cuda.get_device_name(0) if torch.cuda.is_available() else None
print('Project:', PROJECT_DIR)
print('Outputs:', ROOT)
print('GPU:    ', gpu or 'NONE - switch the runtime to GPU before the real run')

## 2b. Check that the code on Drive matches the PC

Every `.py` file on Drive is compared with the version on the PC. If anything is missing or outdated, the cell lists the files to upload. On drive.google.com, upload them into the same folder and choose **Replace** ("Keep both" leaves two files with the same name, and Colab may use the old one).

In [ ]:
# CODE CHECK (generated by tools/update_code_check.py; do not edit by hand)
import hashlib

EXPECTED = {
    'unimodels/__init__.py': '20be044b2c9837cc',
    'unimodels/chemical/__init__.py': '2057b8fd6a9ee2a9',
    'unimodels/chemical/inspect_chemical.py': '3a6dd5a51e8272e7',
    'unimodels/chemical/train_chemical.py': '91e60c32e25657bc',
    'unimodels/common/__init__.py': '594c5c86f6284b2e',
    'unimodels/common/cnn_train.py': '88b7e426f3ad9a85',
    'unimodels/common/config.py': '28086988bcdf6d0a',
    'unimodels/common/inspection.py': '24b5dee3ec724a28',
    'unimodels/common/learning_curves.py': '09e1f3e1a34d61d0',
    'unimodels/common/metrics.py': '4cbcf769694482db',
    'unimodels/common/samples.py': '3b4a8269e61f4c7f',
    'unimodels/common/selection.py': 'e9cb35867e460bd8',
    'unimodels/common/splits.py': '58e1fe0eb0895a52',
    'unimodels/common/tabular.py': 'e4b05a917d796dc9',
    'unimodels/common/tabular_train.py': 'e7e91fe435de5486',
    'unimodels/enose/__init__.py': '348876f0d7a1af71',
    'unimodels/enose/inspect_enose.py': '86d12cac54ae223d',
    'unimodels/enose/train_enose.py': '9975fae0851a25ea',
    'unimodels/image/__init__.py': '1f7d0a9641fda8e4',
    'unimodels/image/cnn.py': 'd9f83cadd83eabc0',
    'unimodels/image/inspect_images.py': '7930b7985c7dc76f',
    'unimodels/image/preprocess_images.py': '68c873a9341286bf',
    'unimodels/image/train_image.py': '5101dd8b31142749',
    'unimodels/image/train_image_masked.py': '1bef66318d1a94b9',
    'unimodels/image_copilot/__init__.py': 'f271f9430b7b8830',
    'unimodels/image_copilot/models.py': '6ebc728dfc735b25',
    'unimodels/image_copilot/train_image_copilot.py': 'd316e0766270545d',
    'unimodels/image_copy/__init__.py': '83d7b3bdd43c1f03',
    'unimodels/image_copy/models_copy.py': 'a3dc56361c6516df',
    'unimodels/image_copy/preprocess_copy.py': '346740607faaaf84',
    'unimodels/image_copy/train_image_copy.py': 'e8af7073dc186f3e',
    'unimodels/image_new/__init__.py': '20caf8f28fb1fe86',
    'unimodels/image_new/augmentation.py': 'd6693e1a219093e7',
    'unimodels/image_new/compare.py': '782ae373b438e65a',
    'unimodels/image_new/data.py': 'e4e481d77a0fd2da',
    'unimodels/image_new/learning_curve.py': 'fcad4d1908b3ff76',
    'unimodels/image_new/models/__init__.py': '7621ed9c7174b17c',
    'unimodels/image_new/models/common.py': 'd44bc6b54a12ee96',
    'unimodels/image_new/models/efficientnet_b0.py': 'bd98312239479851',
    'unimodels/image_new/models/mobilenet_v2.py': 'db909fa14d3552d1',
    'unimodels/image_new/models/resnet18.py': '809976496c315088',
    'unimodels/image_new/train_image_new.py': '7b9c78d27d98c045',
    'unimodels/image_new/tune.py': 'fe73c7614c598e18',
}
problems = []
for rel, digest in EXPECTED.items():
    path = PROJECT_DIR / rel
    if not path.exists():
        problems.append(f'missing   {rel}')
    elif hashlib.sha256(path.read_bytes()).hexdigest()[:16] != digest:
        problems.append(f'outdated  {rel}')
if problems:
    raise RuntimeError(
        'The code on Drive differs from the PC version. Upload these files from the PC '
        '(choose "Replace", never "Keep both"), then run this cell again:\n  ' + '\n  '.join(problems))
print(f'Code on Drive matches the PC version ({len(EXPECTED)} files checked).')

## 3. Install the libraries

In [ ]:
!pip install -q scikit-learn==1.6.1 imbalanced-learn==0.13.0 xgboost==3.2.0 openpyxl

In [ ]:
import sklearn, imblearn, xgboost, torchvision, cv2
print('scikit-learn', sklearn.__version__, '| imbalanced-learn', imblearn.__version__, '| xgboost', xgboost.__version__)
print('torch', torch.__version__, '| torchvision', torchvision.__version__, '| opencv', cv2.__version__)
# Expected first line: scikit-learn 1.6.1 | imbalanced-learn 0.13.0 | xgboost 3.2.0

## 4a. Photo list

Reuses the image branch's inspection (de-duplicated photos, phone from EXIF).

In [ ]:
import json
import pandas as pd
from IPython.display import Image, display
INSP = PROJECT_DIR / 'unimodels' / 'image' / 'results' / 'inspection'
if not (INSP / 'images.csv').exists():
    get_ipython().system(f'python -u -m unimodels.image.inspect_images --data-root "{DATA_ROOT}"')
print('unique photos for the kept samples:', json.loads((INSP / 'inspection_report.json').read_text())['n_images_kept_samples'])

## 4b. Masked-circle crops of all photos

Reuses crops already made (resumable). In FAST mode only one photo per sample.

In [ ]:
flag = '--one-per-sample' if FAST else ''
get_ipython().system(f'python -u -m unimodels.image.preprocess_images --data-root "{DATA_ROOT}" --crop masked {flag}')

## 4c. Augmentation report: training photos per grade before and after balancing

`originals` = real training photos, `copies` = augmented copies added, `photos` = after balancing (equal for every grade). Validation and test keep only real photos. The strip shows one photo of the smallest grade and its first copies (90° rotations, mirror images).

In [ ]:
!python -u -m unimodels.image_new.augmentation --data-root "{DATA_ROOT}" --out "{AUG}" --seeds {SEEDS}

In [ ]:
counts = pd.read_csv(AUG / 'class_counts.csv')
display(counts[counts['seed'] == counts['seed'].min()])
display(Image(str(AUG / 'examples.jpg'), width=1000))

## 5. Before tuning: default hyperparameters, 3 models x 10 seeds

In [ ]:
!python -u -m unimodels.image_new.train_image_new --config default --tasks grade --data-root "{DATA_ROOT}" --out "{OUT_DEFAULT}" {TRAINFLAGS}

In [ ]:
need = [OUT_DEFAULT / 'metadata.json', OUT_DEFAULT / 'grade' / 'best_model.json']
missing = [str(p) for p in need if not p.exists()]
if missing:
    raise RuntimeError('Default training has not finished: missing ' + str(missing) + chr(10) +
                       'Run the training cell again: finished (seed, model) units are reused.')
print('Default training finished:', OUT_DEFAULT)

## 6. Hyperparameter tuning (validation only)

12 trials per model on seeds 42-44 (trial 0 = the default). The best trial per model is chosen on pooled validation macro-F1 with the Brier tie-break. Test data are never used here.

In [ ]:
!python -u -m unimodels.image_new.tune --data-root "{DATA_ROOT}" --out "{TUNING}" {FASTFLAG}

In [ ]:
best = json.loads((TUNING / 'best_hparams.json').read_text())
print('best trial per model:', best['best_trial'])
display(pd.DataFrame(best['best']).T)
trials = pd.read_csv(TUNING / 'trials.csv')
display(trials.sort_values(['model', 'val_pooled_macro_f1'], ascending=[True, False])[['model', 'trial', 'val_pooled_macro_f1', 'val_pooled_brier', 'minutes']].round(3))

## 7. After tuning: the best hyperparameters, 3 models x 10 seeds

In [ ]:
!python -u -m unimodels.image_new.train_image_new --config tuned --tuning-dir "{TUNING}" --tasks grade --data-root "{DATA_ROOT}" --out "{OUT_TUNED}" {TRAINFLAGS}

In [ ]:
need = [OUT_TUNED / 'metadata.json', OUT_TUNED / 'grade' / 'best_model.json']
missing = [str(p) for p in need if not p.exists()]
if missing:
    raise RuntimeError('Tuned training has not finished: missing ' + str(missing) + chr(10) +
                       'Run the training cell again: finished (seed, model) units are reused.')
print('Tuned training finished:', OUT_TUNED)

## 8. Evaluation before vs after hyperparameter tuning

Test set, per physical sample, mean ± std over 10 seeds; macro precision, recall and F1. `selected` = the model chosen on validation for that configuration.

In [ ]:
!python -u -m unimodels.image_new.compare --default "{OUT_DEFAULT}" --tuned "{OUT_TUNED}" --out "{CMP}"

In [ ]:
display(pd.read_csv(CMP / 'before_after_formatted.csv', encoding='utf-8-sig'))
display(Image(str(CMP / 'before_after.png'), width=1100))

## 9. Training samples vs accuracy (best tuned model)

Retrained on 20-100% of the training samples (each subset balanced by augmentation), seeds 42-44. Grey labels = balanced training photos including the augmented copies.

In [ ]:
!python -u -m unimodels.image_new.learning_curve --data-root "{DATA_ROOT}" --results "{OUT_TUNED}" --tuning-dir "{TUNING}" --out "{LC}" {FASTFLAG}

In [ ]:
display(Image(str(LC / 'learning_curve.png'), width=1100))
lc = pd.read_csv(LC / 'learning_curve.csv')
display(lc.groupby('fraction')[['n_train_samples', 'n_balanced_photos', 'train_accuracy', 'val_accuracy', 'test_accuracy', 'test_macro_f1']].mean().round(3))

## 10. Details of the tuned run: selection, per-class results, epoch curves, confusion matrix, Grad-CAM, per phone

In [ ]:
info = json.loads((OUT_TUNED / 'grade' / 'best_model.json').read_text())
print('selected model (validation):', info['best_model'])
display(pd.read_csv(OUT_TUNED / 'grade' / 'selection.csv', index_col=0).round(3))
name = f"{info['best_model']}_test_pooled_{len(info['seeds'])}seeds_per_class.csv"
display(pd.read_csv(OUT_TUNED / 'grade' / 'best_model' / name, index_col=0).round(3))
display(Image(str(OUT_TUNED / 'grade' / 'epoch_curves' / 'all_models_val_accuracy.png'), width=650))
display(Image(str(OUT_TUNED / 'grade' / 'epoch_curves' / f"{info['best_model']}.png"), width=1000))
for png in sorted((OUT_TUNED / 'grade' / 'best_model').glob('*test_pooled*confusion_matrix.png')):
    display(Image(str(png), width=450))
print('Grad-CAM reflection check:', info['gradcam_reflection_check'])
for png in sorted((OUT_TUNED / 'grade' / 'best_model').glob('gradcam_*.png')):
    display(Image(str(png), width=500))
display(pd.read_csv(OUT_TUNED / 'grade' / 'per_phone_metrics.csv'))